# P21 — Mixtral: mezcla dispersa de expertos

## 1. Título y paper

**Paper:** *Mixtral of Experts*  
**Autoría:** Albert Q. Jiang, y otros (Mistral AI)  
**Año y venue:** 2024 · arXiv:2401.04088  
**Nivel:** L3 · **Motor:** `moe`  
**Ficha completa:** [`P21_moe`](../../papers/foundational/P21_moe/README.md)

**Hito:** Desacopla capacidad de cómputo: 47 000 millones de parámetros totales, 13 000 millones activos por token.

- [arXiv:2401.04088](https://arxiv.org/abs/2401.04088)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: En un modelo denso, cada token paga TODOS los parámetros. Crecer en capacidad implica crecer en coste de inferencia en la misma proporción.
2. Ejecutar una implementación mínima de la propuesta: Sustituir la capa feed-forward por 8 expertos con un router que elige 2 por token, y publicar pesos y resultados bajo licencia abierta.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08
- P19
- Shazeer et al. (2017), capa MoE dispersa


## 4. Intuición

En un modelo denso, cada palabra que procesas paga la factura completa del modelo. En una mezcla de expertos hay muchos especialistas pero solo se despiertan dos por token: capacidad de biblioteca, coste de consulta.


## 5. Concepto mínimo

```text
Capa densa:         y = FFN(x)                      coste ∝ todos los parámetros
Capa MoE dispersa:  y = Σ_{i ∈ top-k} g_i(x)·E_i(x)  coste ∝ k expertos

    g(x) = softmax(top-k(x·W_router))
```

En el modelo del paper: 8 expertos, k=2, ≈47 000 M de parámetros totales y ≈13 000 M activos por token.


## 6. Código explicado

El motor enruta 400 tokens con un router top-2 sobre 8 expertos y mide el reparto de carga.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('moe', seed=7)['result']
show(r['parametros'])
print('\ncarga sin balanceo:', r['sin_balanceo']['carga'], '· CV =', r['sin_balanceo']['cv'])
print('carga con balanceo:', r['con_balanceo']['carga'], '· CV =', r['con_balanceo']['cv'])

## 7. Predicción antes de ejecutar

1. ¿Repartirá el router los tokens de forma pareja entre los 8 expertos?
2. Si el 25 % de los parámetros está activo, ¿necesitas el 25 % de la memoria?
3. ¿Qué pasa si un experto no recibe ningún token durante el entrenamiento?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for expertos, k in ((8, 1), (8, 2), (8, 4), (64, 2)):
    fraccion = k / expertos
    print(f'{expertos:>2} expertos, top-{k} → {fraccion:>6.1%} de parámetros activos por token')

## 9. Salida interpretable

El CV (coeficiente de variación de la carga) baja al añadir el término de balanceo. Sin él, unos pocos expertos acaparan los tokens y el resto no recibe gradiente: **se entrena un modelo denso caro disfrazado de disperso**. Ese colapso del router es el fallo característico.


## 10. Comentario pedagógico

La trampa contable: «13 000 M activos» **no** significa que quepa en la memoria de 13 000 M. Hay que cargar los 47 000 M porque cualquier token puede activar cualquier experto. MoE ahorra **cómputo**, no **memoria** — y confundirlo lleva a dimensionar mal el hardware.


## 11. Error o anti-patrón deliberado

Anti-patrón: dimensionar la GPU por los parámetros activos.


In [ ]:
activos_gb = 13e9 * 2 / 1e9
totales_gb = 47e9 * 2 / 1e9
print(f'Presupuesto por parametros ACTIVOS (fp16): {activos_gb:.0f} GB ← INCORRECTO')
print(f'Memoria realmente necesaria (todos)      : {totales_gb:.0f} GB')
print('Comprar una tarjeta por el primer numero es no poder cargar el modelo.')

## 12. Corrección

Las dos cuentas separadas, que es como hay que presupuestar:


In [ ]:
presupuesto = {
    'memoria_pesos_fp16_GB': round(47e9 * 2 / 1e9),
    'computo_por_token_relativo': round(13 / 47, 3),
    'regla': 'MoE ahorra COMPUTO por token, no MEMORIA de pesos',
    'consecuencia': 'mejor throughput por FLOP, misma o mayor factura de VRAM',
}
show(presupuesto)

## 13. Desafío guiado

Comprueba cómo el desbalanceo se agrava al subir el número de expertos.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('moe', seed=semilla)['result']
    print(f"semilla {semilla:>2} · CV sin balanceo {r['sin_balanceo']['cv']:.3f} "
          f"→ con balanceo {r['con_balanceo']['cv']:.3f}")

## 14. Desafío autónomo

Implementa una capa MoE real (expertos = pequeñas MLP) y entrénala en una tarea de clasificación con clases desbalanceadas. Mide la especialización de cada experto y comprueba si el término de balanceo la destruye o la ordena.


## 15. Evidencia de aprendizaje

Guarda la tabla de fracción activa, el CV con y sin balanceo, y el presupuesto de memoria correcto frente al incorrecto.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P21_moe/README.md) · evaluación formal: [`assessments/papers/P21_moe.md`](../../assessments/papers/P21_moe.md)


## 16. Cierre

Ya sabemos abaratar la secuencia y los parámetros. Queda dónde gastar el cómputo que sí queremos gastar: y la respuesta de 2025 fue moverlo al momento de responder.


## 17. Conexión con el siguiente hito

- P22
- modelos abiertos de gran escala

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
